# Notebook 02: Preprocessing and ML Data Preparation

This notebook loads `merged_banner_moodle_with_target.pkl`, removes identifiers and leakage, selects early prediction features, converts data types, creates a stratified train and test split, encodes categories, scales numeric features, and saves the prepared data.


## 1. Import libraries


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

print("Libraries imported successfully.")


## 2. Load the merged dataset


In [ ]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INPUT_FILE = PROJECT_DIR / "data" / "processed" / "merged_banner_moodle_with_target.pkl"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / "ml_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

data = pd.read_pickle(INPUT_FILE)

print("Loaded:", INPUT_FILE)
print("Shape:", data.shape)


## 3. Confirm the target


In [ ]:
TARGET = "academic_risk_label"

if TARGET not in data.columns:
    raise KeyError(f"Target column not found: {TARGET}")

data = data.dropna(subset=[TARGET]).reset_index(drop=True)
data[TARGET] = data[TARGET].astype(int)

display(
    data[TARGET]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET)
    .to_frame("count")
)


## 4. Define excluded and selected features


In [ ]:
IDENTIFIER_COLUMNS = [
    "student_key",
]

FAIRNESS_ONLY_COLUMNS = [
    "gender",
]

LEAKAGE_COLUMNS = [
    "end_term_gpa",
    "end_cgpa",
    "end_term_gpa_range",
    "end_cgpa_range",
    "earned_credits_this_semester",
    "credit_deficit",
    "risk_source",
    "gpa_risk_label",
    "credit_risk_label",
    "risk_condition_count",
    "credits_earned_ratio",
]

NON_PREDICTIVE_COLUMNS = [
    "semester_code",
    "semester_code_banner",
    "semester_code_moodle",
]

NUMERIC_CANDIDATES = [
    "registered_course_count",
    "registered_credits",
    "repeated_course_count_current",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",
    "attendance_rate",
    "absence_count",
    "enrolled_course_count",
    "accessed_course_count",
    "course_access_rate",
    "total_course_event_clicks",
    "total_moodle_course_events",
    "active_days",
    "active_days_rate",
    "zero_activity_days",
    "largest_inactivity_days",
    "learning_material_events",
    "assessment_interaction_events",
]

CATEGORICAL_CANDIDATES = [
    "programme_or_school",
    "programme",
    "school",
    "year_level",
    "previous_term_gpa",
    "previous_term_gpa_band",
    "previous_cgpa",
    "previous_cgpa_band",
    "previous_academic_standing",
]

numeric_features = [c for c in NUMERIC_CANDIDATES if c in data.columns]
categorical_features = [c for c in CATEGORICAL_CANDIDATES if c in data.columns]
selected_features = numeric_features + categorical_features

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal selected features:", len(selected_features))


## 5. Convert numeric and categorical data types


In [ ]:
for column in numeric_features:
    data[column] = (
        data[column]
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
    )
    data[column] = pd.to_numeric(data[column], errors="coerce")

for column in categorical_features:
    data[column] = (
        data[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )
    data[column] = data[column].replace({
        "": pd.NA,
        "NAN": pd.NA,
        "NONE": pd.NA,
    })

display(
    pd.DataFrame({
        "feature": numeric_features,
        "dtype": [str(data[c].dtype) for c in numeric_features],
        "missing_count": [int(data[c].isna().sum()) for c in numeric_features],
        "minimum": [data[c].min() for c in numeric_features],
        "maximum": [data[c].max() for c in numeric_features],
    })
)

display(
    pd.DataFrame({
        "feature": categorical_features,
        "dtype": [str(data[c].dtype) for c in categorical_features],
        "missing_count": [int(data[c].isna().sum()) for c in categorical_features],
        "unique_values": [int(data[c].nunique(dropna=True)) for c in categorical_features],
    })
)


## 6. Create X, y, and fairness data


In [ ]:
if not selected_features:
    raise ValueError("No selected features were found. Check the actual column names.")

X = data[selected_features].copy()
y = data[TARGET].copy()

fairness_data = (
    data[["gender"]].copy()
    if "gender" in data.columns
    else pd.DataFrame(index=data.index)
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fairness data shape:", fairness_data.shape)


## 7. Create stratified training and testing sets


In [ ]:
indices = np.arange(len(data))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

fairness_train = fairness_data.iloc[train_indices].copy()
fairness_test = fairness_data.iloc[test_indices].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class percentage:")
display(
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)

print("Testing class percentage:")
display(
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)


## 8. Build the preprocessing pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Preprocessing pipeline created.")


## 9. Fit on training data and transform both sets


In [ ]:
X_train_prepared_array = preprocessor.fit_transform(X_train)
X_test_prepared_array = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_prepared = pd.DataFrame(
    X_train_prepared_array,
    columns=feature_names,
    index=X_train.index,
)

X_test_prepared = pd.DataFrame(
    X_test_prepared_array,
    columns=feature_names,
    index=X_test.index,
)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared testing shape:", X_test_prepared.shape)
print("Prepared feature count:", len(feature_names))


## 10. Validate the prepared data


In [ ]:
validation = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(X_train_prepared), len(X_test_prepared)],
    "columns": [X_train_prepared.shape[1], X_test_prepared.shape[1]],
    "missing_values": [
        int(X_train_prepared.isna().sum().sum()),
        int(X_test_prepared.isna().sum().sum()),
    ],
})

display(validation)

assert X_train_prepared.shape[1] == X_test_prepared.shape[1]
assert X_train_prepared.isna().sum().sum() == 0
assert X_test_prepared.isna().sum().sum() == 0

print("Prepared data validation completed successfully.")


## 11. Save outputs for Notebook 03


In [ ]:
X_train.to_pickle(OUTPUT_DIR / "X_train_raw.pkl")
X_test.to_pickle(OUTPUT_DIR / "X_test_raw.pkl")

X_train_prepared.to_pickle(OUTPUT_DIR / "X_train_prepared.pkl")
X_test_prepared.to_pickle(OUTPUT_DIR / "X_test_prepared.pkl")

y_train.to_pickle(OUTPUT_DIR / "y_train.pkl")
y_test.to_pickle(OUTPUT_DIR / "y_test.pkl")

fairness_train.to_pickle(OUTPUT_DIR / "fairness_train.pkl")
fairness_test.to_pickle(OUTPUT_DIR / "fairness_test.pkl")

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

pd.DataFrame({
    "feature_name": feature_names
}).to_csv(
    OUTPUT_DIR / "prepared_feature_names.csv",
    index=False,
)

summary = {
    "input_rows": int(len(data)),
    "raw_feature_count": int(len(selected_features)),
    "numeric_feature_count": int(len(numeric_features)),
    "categorical_feature_count": int(len(categorical_features)),
    "prepared_feature_count": int(len(feature_names)),
    "training_rows": int(len(X_train)),
    "testing_rows": int(len(X_test)),
    "random_state": RANDOM_STATE,
}

with open(
    OUTPUT_DIR / "preprocessing_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, indent=2)

print("Saved outputs to:")
print(OUTPUT_DIR)


## Notebook 02 complete

Completed:

1. Loaded the merged dataset.
2. Removed identifiers and leakage from the model inputs.
3. Kept gender separately for fairness evaluation.
4. Selected early Banner, attendance, and Moodle features.
5. Converted numeric and categorical data types.
6. Created stratified training and testing sets.
7. Imputed missing values.
8. Standardised numeric features.
9. One hot encoded categorical features.
10. Saved the prepared data for baseline modelling.
